> 在线讲义：[第 14 章 · 位置编码 positional encoding](https://wikieden.github.io/transformer-visual/lessons/14-positional-encoding.html) · 本书《从零读懂 Transformer》

# 第 14 章 · 位置编码 positional encoding

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/14-positional-encoding.html) · 仓库里 `transformer-deep-dive/notes/lessons/14-positional-encoding.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 不加位置 | 狗咬人 vs 人咬狗 | `bite_output(...)` | 两句"咬"都是 [6.83, 2.63] |
| 整句换顺序 | attention 不认顺序 | `attn(x[perm])` vs `attn(x)[perm]` | 相等：输出只是跟着换顺序 |
| 加号码牌 | $x_i' = x_i + p_i$ | `k + seat_tags` | 狗的分数 4 → 8 |
| 牌一直往上加 | [2, 0] + [0, 1000] | 向量加法 | [2, 1000]：几乎只剩座位号 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import torch
K = {"狗": [2.0, 0.0], "咬": [0.0, 1.0], "人": [1.0, 0.0]}
V = {"狗": [10.0, 0.0], "咬": [2.0, 2.0], "人": [0.0, 10.0]}
Q_BITE = torch.tensor([1.0, 0.0])  # "咬"的 q

**函数 `bite_output`**：'咬'看整句话（省略 ÷√d_k）。seat_tags 是每个座位的号码牌，None 表示不加。

In [ ]:
def bite_output(sentence: list[str], seat_tags: torch.Tensor | None = None) -> tuple[torch.Tensor, torch.Tensor]:
    """'咬'看整句话（省略 ÷√d_k）。seat_tags 是每个座位的号码牌，None 表示不加。"""
    k = torch.tensor([K[w] for w in sentence])
    v = torch.tensor([V[w] for w in sentence])
    q = Q_BITE.clone()
    if seat_tags is not None:
        k = k + seat_tags  # x_i' = x_i + p_i（为了好算，q、k 就等于加完位置后的向量）
        q = q + seat_tags[sentence.index("咬")]
    scores = k @ q
    return scores, torch.softmax(scores, -1) @ v

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
a, b = ["狗", "咬", "人"], ["人", "咬", "狗"]
_, out_a = bite_output(a)
_, out_b = bite_output(b)
print("不加位置：'咬'的输出")
print("  狗咬人:", [round(float(x), 2) for x in out_a], "  人咬狗:", [round(float(x), 2) for x in out_b])
print("  一样吗:", torch.allclose(out_a, out_b))

In [ ]:
print("\n整句一起算也一样：句子换个顺序，输出只是跟着换顺序（每个词的结果不变）")
torch.manual_seed(0)
x = torch.randn(3, 4)
perm = torch.tensor([2, 1, 0])
attn = lambda t: torch.softmax(t @ t.T, -1) @ t
print("  attn(打乱后) == 打乱(attn):", torch.allclose(attn(x[perm]), attn(x)[perm], atol=1e-6))
tags = torch.tensor([[0.0, 1.0], [0.0, 2.0], [0.0, 3.0]])  # 1、2、3 号座的号码牌
sa, _ = bite_output(a, tags)
sb, _ = bite_output(b, tags)

In [ ]:
print("\n加号码牌 [0,1]、[0,2]、[0,3] 以后，'咬'给每个词的分数")
print("  狗咬人（狗、咬、人）:", sa.tolist())
print("  人咬狗（人、咬、狗）:", sb.tolist())
print("  '狗'的分数从", float(sa[0]), "变成", float(sb[2]))

In [ ]:
print("\n号码牌一直往上加：狗坐 1000 号座")
dog = torch.tensor([2.0, 0.0])
print("  ", dog.tolist(), "+ [0, 1000] =", (dog + torch.tensor([0.0, 1000.0])).tolist(), "← 几乎只剩座位号")

## 3. 代码里才看得到的

1. **打乱输入，输出只是跟着打乱**：`attn(x[perm])` 和 `attn(x)[perm]` 完全相等。这叫 permutation equivariance（置换 / 等变：换顺序前后的结果只差一个换顺序），说的就是"attention 只知道袋子里有哪些词"。
2. **`x[perm]` 一行换顺序**：`perm = [2, 1, 0]` 表示新的第 0 行取原来的第 2 行，"狗咬人"就变成"人咬狗"。

## 4. 自己改一改

- 把号码牌换成 [0, 0.1]、[0, 0.2]、[0, 0.3]，"狗"在两句里的分数还不一样吗？差得多还是少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 号码牌跟座位走，谁坐上来谁戴上；
> 牌不能无限变大，不然盖住词本身。